In [57]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [58]:
train_df = pd.read_parquet('train_final.parquet')
test_df = pd.read_parquet('test_final.parquet')
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
train_df.head()

,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,5008159,7,5,417,11632,417,0,59.571429,157.611185,8736,...,32,0.0,0.0,0,0,0.0,0.0,0,0,DoS GoldenEye
1,5004971,7,6,381,11632,381,0,54.428571,144.004464,5792,...,32,0.0,0.0,0,0,0.0,0.0,0,0,DoS GoldenEye
2,5005677,8,6,325,11632,325,0,40.625000,114.904852,5792,...,32,0.0,0.0,0,0,0.0,0.0,0,0,DoS GoldenEye
3,5005163,8,5,537,11632,537,0,67.125000,189.858171,10184,...,32,0.0,0.0,0,0,0.0,0.0,0,0,DoS GoldenEye
4,5006562,8,6,308,11632,308,0,38.500000,108.894444,5840,...,32,0.0,0.0,0,0,0.0,0.0,0,0,DoS GoldenEye


In [59]:
from utils import handle_values
train_df = handle_values(train_df.copy())
test_df = handle_values(test_df.copy())

In [60]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
train_df[' Label']= encoder.fit_transform(train_df[' Label'])
train_df[' Label'].value_counts()

test_df[' Label']= encoder.transform(test_df[' Label'])
test_df[' Label'].value_counts()

 Label
4     6000
0     6000
2     6000
9     6000
3     2059
1     1966
7     1588
10    1179
6     1159
5     1100
8      360
11     301
Name: count, dtype: int64

In [61]:
X_train = train_df.drop(' Label',axis=1)
y_train = train_df[' Label']
X_test = test_df.drop(' Label',axis=1)
y_test = test_df[' Label']

In [62]:
corr = X_train.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col] > 0.9)]

In [63]:
X_train =X_train.drop(to_drop,axis=1)
X_test =X_test.drop(to_drop,axis=1)

In [64]:
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
X_train_np = X_train.values
X_test_np = X_test.values


imputer = SimpleImputer(strategy='constant',fill_value=0)
scaler = RobustScaler()


X_train_np = imputer.fit_transform(X_train_np)
X_train_np = scaler.fit_transform(X_train_np)


X_test_np = imputer.transform(X_test_np)
X_test_np = scaler.transform(X_test_np)


train_final = pd.DataFrame(
    X_train_np, 
    columns=X_train.columns
)
test_final = pd.DataFrame(
    X_test_np, 
    columns=X_test.columns
)

In [65]:
train_final[' Label'] = train_df[' Label']
test_final[' Label'] = test_df[' Label']

In [66]:
train_final.shape

(148278, 44)

In [67]:
y_train.value_counts()

 Label
9     31786
0     30000
4     30000
2     25605
3      8234
7      6350
10     4718
6      4637
5      4399
11     1206
1       983
8       360
Name: count, dtype: int64

In [68]:
import lightgbm as lgb
lgbm_gpu = lgb.LGBMClassifier(
    objective='multiclass', 
    num_class=len(np.unique(y_train)),
    n_estimators=100,            
    device='cpu',           
    n_jobs=-1,                   
    random_state=42,
    min_child_samples=1,
    max_depth=4
)

print("Starting LightGBM training for feature importance using the GPU...")
lgbm_gpu.fit(X_train_np, y_train)
print("Training complete and significantly faster!")

Starting LightGBM training for feature importance using the GPU...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.014381 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6166
[LightGBM] [Info] Number of data points in the train set: 148278, number of used features: 35
[LightGBM] [Info] Start training from score -1.597892
[LightGBM] [Info] Start training from score -5.016235
[LightGBM] [Info] Start training from score -1.756301
[LightGBM] [Info] Start training from score -2.890817
[LightGBM] [Info] Start training from score -1.597892
[LightGBM] [Info] Start training from score -3.517712
[LightGBM] [Info] Start training from score -3.465021
[LightGBM] [Info] Start training from score -3.150634
[LightGBM] [Info] Start training from score -6.020740
[LightGBM] [Info] Start training from score -1.540063
[LightGBM] [Info] Start training from s

In [69]:

# --- C. Get Importance Scores ---
feature_importances = pd.Series(lgbm_gpu.feature_importances_, index=X_train.columns)
feature_importances

 Flow Duration                  800
 Total Fwd Packets              460
Total Length of Fwd Packets     729
 Fwd Packet Length Max          573
 Fwd Packet Length Min          108
 Fwd Packet Length Mean         508
Bwd Packet Length Max           708
 Bwd Packet Length Min          133
Flow Bytes/s                    643
 Flow Packets/s                 384
 Flow IAT Mean                  370
 Flow IAT Std                   438
 Flow IAT Min                   909
 Fwd IAT Mean                   408
 Fwd IAT Min                    834
Bwd IAT Total                   362
 Bwd IAT Mean                   300
 Bwd IAT Std                    253
 Bwd IAT Min                    760
Fwd PSH Flags                   124
 Bwd PSH Flags                    0
 Fwd URG Flags                    2
 Bwd URG Flags                    0
 Bwd Packets/s                  474
 Min Packet Length               76
FIN Flag Count                   61
 RST Flag Count                   0
 PSH Flag Count             

In [70]:
feature_importances_series = pd.Series(
    lgbm_gpu.feature_importances_, 
    index=X_train.columns
).sort_values(ascending=False)

# 2. Print the top 20 to decide on a cutoff

imp_features = feature_importances_series.head(20).index.tolist()
imp_features

['Init_Win_bytes_forward',
 ' Init_Win_bytes_backward',
 ' Flow IAT Min',
 ' Fwd IAT Min',
 ' Flow Duration',
 ' Bwd IAT Min',
 'Total Length of Fwd Packets',
 'Bwd Packet Length Max',
 'Flow Bytes/s',
 ' Fwd Packet Length Max',
 ' Fwd Packet Length Mean',
 ' Bwd Packets/s',
 ' Total Fwd Packets',
 ' Flow IAT Std',
 ' min_seg_size_forward',
 ' Fwd IAT Mean',
 ' Flow Packets/s',
 'Active Mean',
 ' Flow IAT Mean',
 'Bwd IAT Total']

In [71]:
# 1. Filter the importance scores to include ONLY the top 20 features
feature_importances_20 = feature_importances_series[imp_features]

# 2. Calculate the total importance score for these 20 features
# This sum will be used as the denominator for normalization
total_importance = feature_importances_20.sum()

# 3. Normalize the scores to get the percentage weight
# (Divide each score by the total sum and multiply by 100)
normalized_importance = (feature_importances_20 / total_importance) * 100

# 4. Sort and format the results for clear visualization
normalized_importance_sorted = normalized_importance.sort_values(ascending=False).round(2)

print("\n--- Normalized Feature Importance (Weight %) ---")
print(normalized_importance_sorted)


--- Normalized Feature Importance (Weight %) ---
Init_Win_bytes_forward         9.27
 Init_Win_bytes_backward       8.76
 Flow IAT Min                  7.33
 Fwd IAT Min                   6.72
 Flow Duration                 6.45
 Bwd IAT Min                   6.13
Total Length of Fwd Packets    5.88
Bwd Packet Length Max          5.71
Flow Bytes/s                   5.18
 Fwd Packet Length Max         4.62
 Fwd Packet Length Mean        4.09
 Bwd Packets/s                 3.82
 Total Fwd Packets             3.71
 Flow IAT Std                  3.53
 min_seg_size_forward          3.45
 Fwd IAT Mean                  3.29
 Flow Packets/s                3.09
Active Mean                    3.09
 Flow IAT Mean                 2.98
Bwd IAT Total                  2.92
dtype: float64


In [72]:
# from sklearn.feature_selection import SelectKBest, f_classif
# from sklearn.impute import SimpleImputer
# imputer = SimpleImputer(strategy='mean')
# X_imputed = imputer.fit_transform(X_train)

# # Determine the number of columns (features) in your DataFrame
# num_columns = train_df.shape[1]

# # Selecting an appropriate K
# k = min(20, num_columns)  # Will adjust as needed

# # Initialize SelectKBest with the scoring function
# k_best = SelectKBest(score_func=f_classif, k=k)

# # Fit and transform the imputed data to select the top 10 features
# X_new = k_best.fit_transform(X_imputed, y_train)

# # Get the boolean mask of selected features
# selected_features_mask = k_best.get_support()
# elected_feature_names = X_train.columns[selected_features_mask]

# elected_feature_names

In [73]:
final_features = imp_features + [' Label']

train_df_preprocessed = train_final[final_features]
test_df_preprocessed = test_final[final_features]

In [74]:
config_data = {
    'project_name': "Network Anomaly Detection",
    'feature_engineering': {
        'selected_features': imp_features
    }
}

In [75]:
import yaml

with open('config.yaml',"a") as f:
    yaml.dump(config_data,f,default_flow_style=False)

In [76]:
train_df_preprocessed.to_parquet("final_df/train_df_preprocessed2.parquet")
test_df_preprocessed.to_parquet("final_df/test_df_preprocessed2.parquet")

In [77]:
test_df_preprocessed.head()

,Init_Win_bytes_forward,Init_Win_bytes_backward,Flow IAT Min,Fwd IAT Min,Flow Duration,Bwd IAT Min,Total Length of Fwd Packets,Bwd Packet Length Max,Flow Bytes/s,Fwd Packet Length Max,...,Bwd Packets/s,Total Fwd Packets,Flow IAT Std,min_seg_size_forward,Fwd IAT Mean,Flow Packets/s,Active Mean,Flow IAT Mean,Bwd IAT Total,Label
0,0.973298,0.466102,0.563636,1.471338,4.920377,14.155556,0.967320,0.057551,-0.008399,1.333333,...,-0.000186,1.2,1.387643,0.0,2.282925,-0.001222,89.310238,1.539429,163.855894,1
1,-0.027186,0.466102,1052.272727,-0.019108,-0.007078,0.000000,-0.084967,-0.001381,-0.008479,-0.060606,...,0.001287,-0.4,-0.007056,0.0,-0.000536,-0.000194,0.000000,0.011131,0.000000,1
2,0.247608,1.004237,1.945455,6.025478,-0.000776,36.888889,0.588235,0.028085,0.012648,0.779221,...,0.001715,0.2,0.008089,-1.0,0.015506,0.000327,0.000000,-0.004410,0.361417,1
3,-0.027186,1.084746,1.236364,-0.019108,-0.011814,0.000000,-0.065359,0.000000,1.188254,-0.034632,...,1.023615,-0.4,-0.007056,-1.0,-0.000536,0.713066,0.000000,-0.014231,0.000000,1
4,0.247608,1.004237,1.072727,4.726115,-0.001352,37.800000,0.588235,0.028085,0.013811,0.779221,...,0.001820,0.2,0.007299,-1.0,0.014669,0.000413,0.000000,-0.004924,0.342919,1


In [78]:
import joblib

joblib.dump(scaler,'final_df/fitted_robust_scaler.joblib')
joblib.dump(imputer,'final_df/fitted_robust_imputer.joblib')

['final_df/fitted_robust_imputer.joblib']